# Personalized RAG Experiment Notebook

## Overview

## Highlight

## Architecture

## Design Consideration

## Privacy & Safety

## Fallback

## Evaluation Plan

## Future Improvement

## Code

### Environment Setup

### Functions

#### LLM
**Description**  
Use OpenAI LLM client for the LLM call in this project. A simple LLM layer that the subsequent LLM calls, agentic flow will leverage. Open AI API key will be provided in the .env file.



#### python class for chat management


##### Memory python class
Memory class help to manage the different types of memory, including long term/episodic memory that may have separate expiry dates. Given the different memory type, they should have have different expiry period; and for all effective memory for the same user, the chat session should prioritize them differently, say, prioritize episodic memory over long term memory.

Memory should be updated by two different process given 2 different type.

**Slots**
- memory_id
- user_id
- memory_type
- memory_text
- confidence
- expiry_days
- last_update


##### ChatProfile python class
**Description**  
This is a python class that defines the central per-user context object used during a chat session. It contains information from below source
- `Data/user_profiles.csv`: `preferred_style`,`segment`,`interests`,`channel`,`consent_personalization`,`profile_summary`
- `Data/interaction_log.csv`: clicked_docs, all `event_id` under same `uid`, we may need to summarize the view history as one of the background
- `Data/memory_store_seed.json`: all memory class under the same `uid`



##### Query
- query: user query
- preferred_style: from `ChatProfile`
- knowledge_context: knowledge context retrieved from `Retrieval Planner Agent`
- past_relevant_conversation_context: from mem0 RAG
- memory: from `ChatProfile`
- final_prompt: final prompt after consolidating above information and safety check


#### Chat History Management (mem0, RAG)
This is a raw text chat history memory independent from the memory store seed, for chat memory we define a two system management based on the data given and requriement of mem0:
1. raw text chat history (this section), 
2. chat memory (like `Data/memory_store_seed.json`), a summarized memory that separates into episodic and long term memory, then store in the memory_store_seed

**Description of Chat History Management**  
This layer manages user memory in the raw text format using `mem0` library, it will:
1. store each conversation with a unique `event_id`, and store the `uid`
2. store each question & answer as a trunk
3. allows retrieve using RAG with a threshold, if below threshold, report `not found`



#### Knowledge Retrieval (Faiss Database)
**Description**  
This layer use `Data/knowledge_corpus.jsonl` as the source of knowladge, indexed with `category` as filter, will be filtered by user's interest, the content for embedding and semantic search is from `title` and `text`.




#### Agents & Tools

##### Profile Agent
**Description**  
The Profile Agent is responsible for reading user-specific profile information and converting it into a safe, structured session context. given uid, extract below information from source

**Source**
- `Data/user_profiles.csv`: `preferred_style`,`segment`,`interests`,`channel`,`consent_personalization`,`profile_summary`
- `Data/interaction_log.csv`: clicked_docs, all `event_id` under same `uid`



#### Memory Update Agent
**Description**  
Load memory all memory class from given `uid` from `Data/memory_store_seed.json` to form the memory for the user, the data

Memory class help to manage the different types of memory, including long term/episodic memory that may have separate expiry dates. Given the different memory type, they should have have different expiry period; and for all effective memory for the same user, the chat session should prioritize them differently, say, prioritize episodic memory over long term memory.

This assignment will only focus on one establish a software architecture for one session leveraging existing data, the memory update agent in this assignment will only focus on the loading, filtering and prioritization of the memory.

Further offline / batch update strategy will be proposed as next step, instead of coding as it requires iteration with real data.

**Full Memory & Chat history Strategy Proposal**
All chat history will be managed in 3 layer, 1 in raw text and 2 in summarized form as memory seed:
1. raw text for current session, store as a vector database, where each trunk is a qustion and answer pair for reference purpose
2. episodic memory, can be saved as per chat session level, utilized summarization tool to focus on interest, preference and topic, store in the memory store seed
3. long term memory, can be generated based on the episodic memory, focus on interest, preference and topic, store in the memory store seed



#### Retrieval Planner Agent
**Description**  
Given user's query and user's interest from `ChatProfile` to perform RAG and produce knowledge context

The knowledge retrival will take a 3 step approach, this gives personalized context a chance to be select in the final but still based on the relevance at the end:
1. retrieve context given personalized field such as interest and reading history, select top 3
2. retrieve context without filter, select top 3
3. dedup 1 & 2 and select top 3



#### Safety Agents
Separate to below two agents

**Description**  
To guardrail the query and context to not include
1. irrelevant questions that is not about financial advise
2. any sensitive information in the context or query

As the provided data seems does not contain sensitive information, guardrail sensitive information including identifier, account, monetary information in this safety agent to exclude them from the prompt.



#### Prompt Construction
**Description**  
Processing all the information in `Query` class pick relevant information from preference, knowledge_context, chat_history_context, safety check then form the final prompt, use `DSPy` library to optimize



#### Agentic Flow - Orchestration
**Description**
Use LangGraph to form the entire orchestration layer, I can describe the steps as below:
1. when session start, load `ChatProfile` using Profile Agent and Memory Agent to populate `ChatProfile` 
2. for every query, create `Query` class, populate the necessary info from `ChatProfile`
3. use `Retrieval Planner Agent` to populate the context
4. conduct Safety check with `Safety Agent`
5. prompt construction
6. send to OpenAI LLM for generation
7. update chat history with mem0

### Load Session Data

### Test Run - Generation

- loading the config_template.json